# Synthetic Dataset Regeneration

This notebook regenerates only the rows needed after cleanup.

Workflow:
1. Load the cleaned dataset exported by the repair notebook
2. Load the inferred term policy and post-cleanup label gaps
3. Decide which missing rows should be term-conditioned vs generic Singapore-context rows
4. Generate only the missing rows
5. Export regenerated files for re-evaluation in the judge notebook

In [25]:
import os
import json
import re
import time
from pathlib import Path
from collections import Counter, defaultdict
from dotenv import load_dotenv
from google import genai

REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

GEMINI_KEY = os.getenv('GEMINI_KEY')
GEMINI_MODEL = 'gemini-3-flash-preview'
if not GEMINI_KEY:
    raise ValueError('GEMINI_KEY not found in .env')
GENAI_CLIENT = genai.Client(api_key=GEMINI_KEY)

DATA_DIR = REPO_ROOT / 'data' / 'singlish' / 'synthetic' / '20-terms'
REPAIR_DIR = DATA_DIR / 'repair_outputs'
CLEANED_SPLIT_DIR = REPAIR_DIR / 'cleaned_splits'
GROUND_TRUTH_FILE = DATA_DIR / 'chosen-20-terms.json'
TERM_POLICY_FILE = REPAIR_DIR / 'term_policy.json'
POST_CLEANUP_SUMMARY_FILE = REPAIR_DIR / 'post_cleanup_summary.json'
REMOVED_HATE_BLUEPRINTS_FILE = REPAIR_DIR / 'removed_hate_blueprints.json'
REGEN_OUTPUT_DIR = REPAIR_DIR / 'regenerated_outputs'
REGEN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_GROUP_DESCRIPTIONS = {
    'chinese': 'Chinese Singaporeans',
    'malay': 'Malay Singaporeans',
    'indian': 'Indian Singaporeans',
    'eurasian': 'Eurasian Singaporeans',
    'tamil': 'Tamil community in Singapore',
    'hindi': 'Hindi-speaking community in Singapore',
    'punjabi': 'Punjabi / Sikh community in Singapore',
    'migrant': 'Migrant workers in Singapore (general)',
    'worker': 'Foreign workers in Singapore (general)',
    'bangladeshi': 'Bangladeshi workers in Singapore',
    'bangla': 'Bangladeshi community (colloquial term)',
    'filipino': 'Filipino workers / community in Singapore',
    'vietnamese': 'Vietnamese workers / community in Singapore',
    'myanmar': 'Myanmar workers / community in Singapore',
    'burmese': 'Burmese workers / community in Singapore',
    'prc': 'Chinese nationals / PRCs in Singapore',
    'muslim': 'Muslim community in Singapore',
    'islam': 'Islamic community in Singapore',
    'christian': 'Christian community in Singapore',
    'buddhist': 'Buddhist community in Singapore',
    'hindu': 'Hindu community in Singapore',
    'sikh': 'Sikh community in Singapore',
    'taoist': 'Taoist community in Singapore',
    'foreigner': 'Foreigners in Singapore (general)',
    'foreign_worker': 'Foreign workers in Singapore',
}

TARGET_GROUP_KEYWORDS = {
    'chinese': ['chinese', 'cina babi'],
    'malay': ['malay'],
    'indian': ['indian', 'keling', 'anneh', 'apu neh neh'],
    'eurasian': ['eurasian'],
    'tamil': ['tamil'],
    'hindi': ['hindi'],
    'punjabi': ['punjabi', 'sikh'],
    'migrant': ['migrant'],
    'worker': ['worker'],
    'bangladeshi': ['bangladeshi'],
    'bangla': ['bangla', 'bangladeshi'],
    'filipino': ['filipino', 'pinoy'],
    'vietnamese': ['vietnamese'],
    'myanmar': ['myanmar'],
    'burmese': ['burmese', 'myanmar'],
    'prc': ['prc', 'china', 'chinese national'],
    'muslim': ['muslim'],
    'islam': ['islam', 'islamic', 'muslim'],
    'christian': ['christian'],
    'buddhist': ['buddhist'],
    'hindu': ['hindu'],
    'sikh': ['sikh'],
    'taoist': ['taoist', 'taoism'],
    'foreigner': ['foreigner', 'foreign'],
    'foreign_worker': ['foreign worker', 'foreign'],
}

with open(GROUND_TRUTH_FILE, encoding='utf-8') as f:
    ground_truth_list = json.load(f)
TERM_LOOKUP = {entry['term'].casefold(): entry for entry in ground_truth_list}

with open(TERM_POLICY_FILE, encoding='utf-8') as f:
    term_policy_rows = json.load(f)
policy_by_term = {row['term']: row['policy'] for row in term_policy_rows}
identity_linked_terms = {row['term'] for row in term_policy_rows if row.get('identity_linked')}

with open(POST_CLEANUP_SUMMARY_FILE, encoding='utf-8') as f:
    post_cleanup_summary = json.load(f)

with open(REMOVED_HATE_BLUEPRINTS_FILE, encoding='utf-8') as f:
    removed_hate_blueprints = json.load(f)

active_target_groups = sorted({row['target_group'] for row in removed_hate_blueprints if row.get('target_group')})
SG_TARGET_GROUPS = {group: TARGET_GROUP_DESCRIPTIONS[group] for group in active_target_groups if group in TARGET_GROUP_DESCRIPTIONS}
GROUP_KEYWORDS = {group: TARGET_GROUP_KEYWORDS.get(group, [group.replace('_', ' ')]) for group in active_target_groups}

print('Gemini model:', GEMINI_MODEL)
print('Repair directory:', REPAIR_DIR)
print('Regeneration output dir:', REGEN_OUTPUT_DIR)
print('Removed hate blueprints:', len(removed_hate_blueprints))
print('Active hate target groups:', active_target_groups)

Gemini model: gemini-3-flash-preview
Repair directory: /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs
Regeneration output dir: /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/regenerated_outputs
Removed hate blueprints: 24
Active hate target groups: ['chinese', 'indian', 'malay']


## Load cleaned dataset and compute regeneration targets

In [26]:
cleaned_rows = []
for split_name in ['train', 'val', 'test']:
    split_file = CLEANED_SPLIT_DIR / f'{split_name}.jsonl'
    if not split_file.exists():
        raise FileNotFoundError(f'Missing cleaned split: {split_file}')
    with open(split_file, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row['split'] = split_name
            cleaned_rows.append(row)

original_rows = []
for split_name in ['train', 'val', 'test']:
    split_file = DATA_DIR / f'{split_name}.jsonl'
    with open(split_file, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row['split'] = split_name
            original_rows.append(row)

current_label_counts = Counter(r['label'] for r in cleaned_rows)
target_label_counts = Counter(r['label'] for r in original_rows)
label_gap = {
    label: target_label_counts.get(label, 0) - current_label_counts.get(label, 0)
    for label in ['hate', 'offensive', 'normal']
}

target_split_label_counts = defaultdict(Counter)
for row in original_rows:
    target_split_label_counts[row['split']][row['label']] += 1

current_split_label_counts = defaultdict(Counter)
for row in cleaned_rows:
    current_split_label_counts[row['split']][row['label']] += 1

split_label_gap = defaultdict(dict)
for split_name in ['train', 'val', 'test']:
    for label in ['hate', 'offensive', 'normal']:
        split_label_gap[split_name][label] = (
            target_split_label_counts[split_name][label] - current_split_label_counts[split_name][label]
        )

hate_target_gap_by_split = post_cleanup_summary.get('hate_target_gap_by_split', {})

print('Cleaned rows:', len(cleaned_rows))
print('Current label counts:', current_label_counts)
print('Target label counts:', target_label_counts)
print('Global label gaps:', label_gap)
print('\nSplit-level gaps:')
for split_name in ['train', 'val', 'test']:
    print(split_name, split_label_gap[split_name])
print('\nHate target-group gaps by split:')
for split_name in ['train', 'val', 'test']:
    print(split_name, hate_target_gap_by_split.get(split_name, {}))

Cleaned rows: 370
Current label counts: Counter({'hate': 156, 'normal': 127, 'offensive': 87})
Target label counts: Counter({'hate': 180, 'normal': 180, 'offensive': 180})
Global label gaps: {'hate': 24, 'offensive': 93, 'normal': 53}

Split-level gaps:
train {'hate': 8, 'offensive': 39, 'normal': 20}
val {'hate': 7, 'offensive': 25, 'normal': 18}
test {'hate': 9, 'offensive': 29, 'normal': 15}

Hate target-group gaps by split:
train {'chinese': 2, 'indian': 2, 'malay': 4}
val {'chinese': 2, 'indian': 4, 'malay': 1}
test {'chinese': 2, 'indian': 3, 'malay': 4}


## Build regeneration tasks
This proposes what to regenerate, using the inferred term policy from the repair notebook.

In [27]:
def extract_context(entry):
    definitions = (entry.get('definitions') or '').strip()
    usage_notes = (entry.get('usage_notes') or '').strip()
    categories = entry.get('categories') or []
    parts = []
    if categories:
        parts.append(f"Categories: {', '.join(categories)}")
    if definitions:
        parts.append(f"Definitions:\n{definitions}")
    if usage_notes:
        parts.append(f"Usage notes:\n{usage_notes}")
    text = '\n\n'.join(parts).strip()
    return text[:1500] + ('\n...' if len(text) > 1500 else '')


def choose_candidate_terms(label):
    candidate_terms = sorted(policy_by_term)
    if label in {'normal', 'offensive'}:
        candidate_terms = [t for t in candidate_terms if policy_by_term[t] not in {'hate_only_candidate'}]
        # Exclude identity-linked terms: their non-hate use still evokes the target group
        candidate_terms = [t for t in candidate_terms if t not in identity_linked_terms]
    return candidate_terms or sorted(policy_by_term)


regeneration_tasks = []

# Hate regeneration is targeted from the exact deleted hate rows.
for idx, blueprint in enumerate(removed_hate_blueprints):
    term = blueprint['source_term']
    regeneration_tasks.append(
        {
            'task_id': f"{blueprint['split']}:hate:{idx}",
            'split': blueprint['split'],
            'label': 'hate',
            'target_group': blueprint['target_group'],
            'source_term': term,
            'term_policy': policy_by_term.get(term, 'hate_dominant'),
            'prompt_style': 'term_conditioned_hate',
            'term_context': extract_context(TERM_LOOKUP[term.casefold()]),
            'regen_source': 'deleted_hate_blueprint',
            'blueprint_row_id': blueprint.get('row_id'),
        }
    )

# Non-hate regeneration remains split/label gap driven.
for split_name in ['train', 'val', 'test']:
    for label in ['offensive', 'normal']:
        gap = split_label_gap[split_name][label]
        if gap <= 0:
            continue

        candidate_terms = choose_candidate_terms(label)
        for idx in range(gap):
            term = candidate_terms[idx % len(candidate_terms)]
            policy = policy_by_term[term]
            if policy in {'hate_only_candidate', 'hate_dominant'} or term in identity_linked_terms:
                prompt_style = 'generic_singlish_non_hate' if label == 'offensive' else 'generic_singlish_normal'
            else:
                prompt_style = 'term_conditioned_non_hate'

            regeneration_tasks.append(
                {
                    'task_id': f'{split_name}:{label}:{idx}',
                    'split': split_name,
                    'label': label,
                    'target_group': 'none',
                    'source_term': term,
                    'term_policy': policy,
                    'prompt_style': prompt_style,
                    'term_context': extract_context(TERM_LOOKUP[term.casefold()]),
                    'regen_source': 'split_label_gap',
                }
            )

print('Regeneration task counts by split/label:')
summary = defaultdict(Counter)
for task in regeneration_tasks:
    summary[task['split']][task['label']] += 1
for split_name in ['train', 'val', 'test']:
    print(split_name, dict(summary[split_name]))

print('\nHate tasks by target group:')
hate_group_summary = defaultdict(Counter)
for task in regeneration_tasks:
    if task['label'] == 'hate':
        hate_group_summary[task['split']][task['target_group']] += 1
for split_name in ['train', 'val', 'test']:
    print(split_name, dict(hate_group_summary[split_name]))

print('\nPrompt styles:')
print(Counter(task['prompt_style'] for task in regeneration_tasks))
print('\nSample tasks:')
for task in regeneration_tasks[:15]:
    print(task['task_id'], '|', task['split'], '|', task['label'], '|', task['source_term'], '|', task['term_policy'], '|', task['prompt_style'], '|', task['target_group'], '|', task['regen_source'])

Regeneration task counts by split/label:
train {'hate': 8, 'offensive': 39, 'normal': 20}
val {'hate': 7, 'offensive': 25, 'normal': 18}
test {'hate': 9, 'offensive': 29, 'normal': 15}

Hate tasks by target group:
train {'indian': 2, 'malay': 4, 'chinese': 2}
val {'indian': 4, 'chinese': 2, 'malay': 1}
test {'malay': 4, 'indian': 3, 'chinese': 2}

Prompt styles:
Counter({'term_conditioned_non_hate': 138, 'term_conditioned_hate': 24, 'generic_singlish_non_hate': 5, 'generic_singlish_normal': 3})

Sample tasks:
train:hate:0 | train | hate | apu neh neh | hate_only_candidate | term_conditioned_hate | indian | deleted_hate_blueprint
train:hate:1 | train | hate | big fuck | offensive_dominant | term_conditioned_hate | malay | deleted_hate_blueprint
train:hate:2 | train | hate | bo lan pa | offensive_dominant | term_conditioned_hate | malay | deleted_hate_blueprint
train:hate:3 | train | hate | CB kia | offensive_dominant | term_conditioned_hate | malay | deleted_hate_blueprint
train:hate:4 

## Prompt builders and Gemini generation
These prompts are tailored to the task style chosen for each regeneration task.

In [35]:
REGEN_RESPONSE_SCHEMA = {
    'type': 'object',
    'properties': {
        'text': {'type': 'string'},
        'label': {'type': 'string', 'enum': ['hate', 'offensive', 'normal']},
        'target_group': {'type': 'string'},
    },
    'required': ['text', 'label', 'target_group'],
}


def build_regeneration_prompt(task):
    term = task['source_term']
    label = task['label']
    term_context = task['term_context']
    target_group = task['target_group']
    prompt_style = task['prompt_style']

    if prompt_style == 'term_conditioned_hate':
        group_desc = SG_TARGET_GROUPS[target_group]
        return f'''Generate exactly ONE JSON object for a Singapore-context hate speech dataset.

Term: "{term}"
Context:
{term_context}

Requirements:
- label must be "hate"
- target_group must be "{target_group}"
- text must include the exact term "{term}"
- text must explicitly target {group_desc}
- text should sound like plausible Singapore online language

Output format:
{{"text": "...", "label": "hate", "target_group": "{target_group}"}}

Output only JSON.'''

    if prompt_style == 'term_conditioned_non_hate':
        return f'''Generate exactly ONE JSON object for a Singapore-context dataset.

Term: "{term}"
Context:
{term_context}

Requirements:
- label must be "{label}"
- target_group must be "none"
- text must include the exact term "{term}"
- if label is offensive: insulting or crude but not hate toward a protected group
- if label is normal: neutral, quoted, descriptive, or meta-linguistic use
- text should sound like plausible Singapore online language

Output format:
{{"text": "...", "label": "{label}", "target_group": "none"}}

Output only JSON.'''

    if prompt_style == 'generic_singlish_non_hate':
        return f'''Generate exactly ONE JSON object for a Singapore-context dataset.

Requirements:
- label must be "offensive"
- target_group must be "none"
- do NOT use the term "{term}" if it would sound unnatural
- text should be offensive, insulting, or vulgar in Singapore-context language
- text must NOT target a protected identity group

Output format:
{{"text": "...", "label": "offensive", "target_group": "none"}}

Output only JSON.'''

    return f'''Generate exactly ONE JSON object for a Singapore-context dataset.

Requirements:
- label must be "normal"
- target_group must be "none"
- do NOT use the term "{term}" if it would sound unnatural
- text should be neutral, ordinary, realistic Singapore-context language
- no hate, no strong insult, no protected-group targeting

Output format:
{{"text": "...", "label": "normal", "target_group": "none"}}

Output only JSON.'''


def parse_regeneration_response(raw):
    if not raw:
        return None
    text = raw.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(data, list):
        data = data[0] if data else None
    if not isinstance(data, dict):
        return None
    data['label'] = str(data.get('label', '')).strip().lower()
    data['target_group'] = str(data.get('target_group', 'none')).strip().lower()
    data['text'] = str(data.get('text', '')).strip()
    if data['label'] not in {'hate', 'offensive', 'normal'}:
        return None
    if not data['text']:
        return None
    return data


def call_gemini_json(prompt, temperature=0.7, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            response = GENAI_CLIENT.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config={
                    'temperature': temperature,
                    'top_p': 0.9,
                    'response_mime_type': 'application/json',
                    'response_json_schema': REGEN_RESPONSE_SCHEMA,
                },
            )
            return (getattr(response, 'text', None) or '').strip()
        except Exception as exc:
            err = str(exc)
            transient = any(k in err for k in ['503', '429', 'UNAVAILABLE', 'RESOURCE_EXHAUSTED', 'overloaded', 'rate limit'])
            if transient and attempt < max_retries:
                wait = min(2 ** attempt, 120)
                print(f'  retry {attempt}/{max_retries} after error: {err[:120]}')
                time.sleep(wait)
            else:
                print(f'  failed after attempt {attempt}: {err[:200]}')
                return None


def term_appears_in_text(term, text):
    text_lower = (text or '').lower()
    term_lower = term.lower().strip()

    if term_lower in text_lower:
        return True

    clean_term = term_lower.replace('\u2026', '').replace('...', '')
    core_words = [word for word in clean_term.split() if len(word) > 1]
    if core_words:
        matches = sum(1 for word in core_words if word in text_lower)
        if matches == len(core_words):
            return True

    stop_words = {'buay', 'the', 'and', 'for', 'a', 'is', 'to', 'of', 'in'}
    words = [
        word for word in term_lower.split()
        if len(word) > 2 and word not in stop_words and '.' not in word and '\u2026' not in word
    ]
    if words:
        matches = sum(1 for word in words if word in text_lower)
        if matches >= max(1, len(words) * 0.6):
            return True

    if term.isupper() and len(term) <= 6 and term in (text or ''):
        return True

    return False


def has_group_mention(text, target_group):
    tg = (target_group or '').strip().lower()
    if tg in ('', 'none'):
        return True
    keywords = GROUP_KEYWORDS.get(tg, [tg.replace('_', ' ')])
    text_lower = (text or '').lower()
    return any(keyword in text_lower for keyword in keywords)


def validate_regenerated_row(task, row):
    if not row:
        return False, 'parse_failed'
    if row['label'] != task['label']:
        return False, f'label_mismatch:{row["label"]}'
    if task['prompt_style'] in {'term_conditioned_hate', 'term_conditioned_non_hate'}:
        if not term_appears_in_text(task['source_term'], row['text']):
            return False, 'missing_source_term'
    if task['label'] == 'hate':
        if row['target_group'] != task['target_group']:
            return False, f'target_group_mismatch:{row["target_group"]}'
        if not has_group_mention(row['text'], row['target_group']):
            return False, 'hate_missing_target_group_mention'
    else:
        row['target_group'] = 'none'
    return True, 'ok'

print('Prompt builders ready')

Prompt builders ready


## Run regeneration
This executes only the missing tasks and writes regenerated rows to a JSONL file.

In [38]:
REGEN_RESULTS_FILE = REGEN_OUTPUT_DIR / 'regenerated_rows.jsonl'
RESET_REGEN_RESULTS = False

if RESET_REGEN_RESULTS and REGEN_RESULTS_FILE.exists():
    REGEN_RESULTS_FILE.unlink()
    print('Deleted previous regeneration results')

completed_task_ids = set()
if REGEN_RESULTS_FILE.exists():
    with open(REGEN_RESULTS_FILE, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                completed_task_ids.add(json.loads(line)['task_id'])
            except Exception:
                pass

print('Existing completed tasks:', len(completed_task_ids))
print('Pending tasks:', sum(1 for t in regeneration_tasks if t['task_id'] not in completed_task_ids))

written = 0
failed = 0
with open(REGEN_RESULTS_FILE, 'a', encoding='utf-8') as fh:
    for idx, task in enumerate(regeneration_tasks, start=1):
        if task['task_id'] in completed_task_ids:
            continue
        prompt = build_regeneration_prompt(task)
        raw = call_gemini_json(prompt)
        parsed = parse_regeneration_response(raw)
        ok, reason = validate_regenerated_row(task, parsed)
        if ok:
            out = {
                'task_id': task['task_id'],
                'split': task['split'],
                'label': task['label'],
                'target_group': parsed['target_group'],
                'source_term': task['source_term'],
                'term_policy': task['term_policy'],
                'prompt_style': task['prompt_style'],
                'text': parsed['text'],
            }
            fh.write(json.dumps(out, ensure_ascii=False) + '\n')
            fh.flush()
            written += 1
        else:
            failed += 1
            print(f'[{idx}/{len(regeneration_tasks)}] failed {task["task_id"]}: {reason}')
        if idx % 10 == 0:
            print(f'progress: {idx}/{len(regeneration_tasks)} | written={written} failed={failed}')
        time.sleep(0.15)

print('Regeneration complete')
print('Written:', written)
print('Failed:', failed)
print('Output:', REGEN_RESULTS_FILE)

Existing completed tasks: 169
Pending tasks: 1
progress: 150/170 | written=1 failed=0
Regeneration complete
Written: 1
Failed: 0
Output: /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/regenerated_outputs/regenerated_rows.jsonl


## Merge regenerated rows into repaired splits
This combines the cleaned dataset with regenerated rows and writes final repaired split files.

In [40]:
regenerated_rows = []
if REGEN_RESULTS_FILE.exists():
    with open(REGEN_RESULTS_FILE, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                regenerated_rows.append(json.loads(line))

final_rows_by_split = defaultdict(list)
for row in cleaned_rows:
    final_rows_by_split[row['split']].append(
        {
            'text': row['text'],
            'label': row['label'],
            'target_group': row.get('target_group', 'none'),
            'source_term': row['source_term'],
            'source_category': row.get('source_category', ''),
        }
    )

# Dedup: skip any regenerated row whose text already exists in the cleaned set
existing_texts = {row['text'] for row in cleaned_rows}
regen_dupes = 0
for row in regenerated_rows:
    if row['text'] in existing_texts:
        regen_dupes += 1
        continue
    existing_texts.add(row['text'])  # guard against dupes within the regen batch too
    final_rows_by_split[row['split']].append(
        {
            'text': row['text'],
            'label': row['label'],
            'target_group': row.get('target_group', 'none'),
            'source_term': row['source_term'],
            'source_category': '',
        }
    )
if regen_dupes:
    print(f'Skipped {regen_dupes} regenerated rows that duplicated existing cleaned rows')

FINAL_SPLIT_DIR = REGEN_OUTPUT_DIR / 'repaired_splits'
FINAL_SPLIT_DIR.mkdir(parents=True, exist_ok=True)
for split_name in ['train', 'val', 'test']:
    output_path = FINAL_SPLIT_DIR / f'{split_name}.jsonl'
    with open(output_path, 'w', encoding='utf-8') as f:
        for row in final_rows_by_split[split_name]:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(f'Wrote {len(final_rows_by_split[split_name])} rows -> {output_path}')

print('Final label counts:')
print(Counter(row['label'] for split in final_rows_by_split.values() for row in split))

Skipped 1 regenerated rows that duplicated existing cleaned rows
Wrote 180 rows -> /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/regenerated_outputs/repaired_splits/train.jsonl
Wrote 180 rows -> /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/regenerated_outputs/repaired_splits/val.jsonl
Wrote 180 rows -> /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/regenerated_outputs/repaired_splits/test.jsonl
Final label counts:
Counter({'hate': 180, 'normal': 180, 'offensive': 180})


## Export regeneration plan
This writes the regeneration task list for review before implementing generation prompts.

In [31]:
regeneration_tasks_file = REGEN_OUTPUT_DIR / 'regeneration_tasks.json'
with open(regeneration_tasks_file, 'w', encoding='utf-8') as f:
    json.dump(regeneration_tasks, f, ensure_ascii=False, indent=2)

print('Wrote regeneration plan ->', regeneration_tasks_file)
print('Total tasks:', len(regeneration_tasks))

Wrote regeneration plan -> /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/regenerated_outputs/regeneration_tasks.json
Total tasks: 170
